# Sections 1 and 2: Positioning and Direct Replication

ECON 5200 Midterm Project · Replication Study

> **🛑 AI policy, and it is not the problem sets' policy.** Sections 1 and 2 — the
> positioning and the direct replication — are **no AI**. Not for the code, not for
> the prose. Making a computer agree with a published table is the exercise that
> establishes you can read a paper, and a model that has already read the paper will
> simply hand you the answer without your ever finding the discrepancy. Sections 3 to
> 6 authorise AI as a documented co-pilot under **P.R.I.M.E.**, on the same terms as
> every problem set: you file the full transcript in `ai-appendix.md`, and if you
> cannot explain a line you cannot submit it. State in your README which sections you
> used it in. This follows the course rule — manual command first, expansion second —
> and using AI in Sections 1 or 2 is an integrity violation, not a style disagreement.

## Setup

Imports and paths only.

In [15]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# The repository root, whether this runs from notebooks/ (Jupyter) or from the root (reproduce.py).
ROOT = Path.cwd() if (Path.cwd() / "download_data.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
(ROOT / "output").mkdir(exist_ok=True)

from download_data import MY_PAPER, PAPERS, fetch

## The data

`hansen_dwi.dta` has one row per DUI stop in Washington State, 1999–2007: 214,558 stops. The variables the design
uses:

| variable | what it is |
|---|---|
| `recidivism` | 1 if the driver was stopped again for drunk driving within the paper's follow-up window (the outcome) |
| `bac1`, `bac2` | the two breath-test readings, in grams per decilitre (0.08 is the DUI threshold, 0.15 the aggravated one) |
| `low_score` | the lower of the two readings, **in thousandths** (80 means 0.080) |
| `male`, `white`, `aged`, `acc` | sex, race, age, and whether the stop involved an accident |
| `year`, `Date` | when the stop happened |

Two decisions are yours and belong in Section 2's discrepancy table: **which reading is the running variable**
(the paper says which one the law uses; find it), and how the paper's bandwidth, kernel and controls are set.

In [13]:
# Set MY_PAPER once, in download_data.py. fetch() downloads into data/ only if a file is missing.
if MY_PAPER:
    paths = fetch(MY_PAPER)
    print(PAPERS[MY_PAPER]["paper"])
    # TODO: read the file(s), e.g. df = pd.read_stata(paths[0])
    dwi = pd.read_stata(paths[0])
    print(dwi.shape)
    print(dwi["year"].min(), dwi["year"].max())
else:
    print("TODO: set MY_PAPER in download_data.py (castle, card, lee, dw, hansen or abortion).")

  data/hansen_dwi.dta  9.5 MB
Hansen (2015), punishment and deterrence in drunk driving
(214558, 12)
1999.0 2007.0


## Section 1: Positioning (12 pts)

Two pages maximum in the report. What question the paper asks, what was believed before it, what it changed, and how it has been received, with at least three citations to work that engages with it critically. Then the identification strategy and the assumption it rests on, in your own words.

The prose goes in the report. Keep your notes and the three citations here.

**Critical citations (at least three):**

1. Dušek, L. and Traxler, C. (2022). "Learning from Law Enforcement." *Journal of the European Economic Association* 20(2): 739–777. They see Hansen as the closest paper to theirs, but say a DUI also takes away the driver's license. So Hansen can't fully tell deterrence apart from incapacitation.
2. Gehrsitz, M. (2017). "Speeding, Punishment, and Recidivism: Evidence from a Regression Discontinuity Design." *Journal of Law and Economics* 60(3): 497–528. He says Hansen only gets an intent-to-treat effect, since not everyone above 0.08 gets the full penalty. It is also unclear which part of the penalty causes the drop.
3. Sloan, F. A. (2020). "Drinking and Driving." In K. F. Zimmermann (ed.), *Handbook of Labor, Human Resources and Population Economics*. Springer, pp. 1–31. He accepts Hansen's design, but notes some drivers may keep driving without a license, which the data can't show. So the deterrence story is less certain.
4. Kolesár, M. and Rothe, C. (2018). "Inference in Regression Discontinuity Designs with a Discrete Running Variable." *American Economic Review* 108(8): 2277–2304. The article says that clustering standard errors by the values of a discrete running variable does not fix a wrong functional form and can give poor coverage. Hansen's BAC moves in steps of 0.001 and he clusters by these bins, so his confidence intervals may be less reliable than they look.

**The identification strategy and its assumption, in my own words:**



Hansen uses an RD at BAC = 0.08. Drivers just below and above 0.08 are almost the same, but the punishment jumps at 0.08. So the jump in repeat drunk driving at 0.08 is that reduced form effect of the punishment. The key assumptions is that nothing else changes suddenly at 0.08, which seems reasonable since drivers can't control their BAC and there is no bunching. The effect is local to drivers near 0.08, which is potential treat, and some may come from license loss, not only deterrence.

## Section 2: Direct Replication (24 pts)

Reproduce the paper's headline result and report your number beside theirs. You will not match exactly. Document every discrepancy and its likely source: sample construction, variable definition, software defaults, a decision the paper does not describe. Where the paper is ambiguous about a choice, say so: those ambiguities are your Section 3.

In [21]:
CUTOFF = 0.08

In [22]:
dwi["bac_min"] = dwi["low_score"].astype(float) / 1000

RUNNING = "bac_min"
H = 0.05
KERNEL = "rectangular"
CONTROLS = ["male", "white", "aged", "acc"]

s = dwi[(dwi[RUNNING] >= CUTOFF - H) & (dwi[RUNNING] <= CUTOFF + H)].copy()
s["x"] = s[RUNNING] - CUTOFF
s["dui"] = (s[RUNNING] >= CUTOFF).astype(int)

rd_fit = smf.ols("recidivism ~ dui + x + dui:x + male + white + aged + acc + C(year)", data=s).fit(cov_type="cluster", cov_kwds={"groups": s["low_score"]})
JUMP = float(rd_fit.params["dui"])
print(JUMP, rd_fit.bse["dui"], len(s))

-0.022163327949850046 0.004141808892258096 93899


### Your number, with its unit

`recidivism` is 0 or 1, so its mean is a proportion and your jump is a change in a proportion. Read it in
**percentage points**: −0.02 is minus 2 percentage points, not "minus 2 percent". If you want a relative change,
name its base: the mean recidivism just below the cutoff, within your bandwidth. Fill all five names:

- `PRINTED`: the jump exactly as your code prints it
- `IN_UNITS`: the same jump in the unit you name in `UNIT`
- `UNIT`: one of `percentage points`, `percent relative to the mean below the cutoff`
- `PAPER_NUMBER`, `PAPER_WHERE`: the paper's own estimate, and its table, column and row, read from the paper

In [24]:
# YOUR TURN — Section 2, no AI
PRINTED = JUMP
IN_UNITS = 100 * JUMP
UNIT = "percentage points"
PAPER_NUMBER = -0.021
PAPER_WHERE = "Table 3, Panel A (bandwidth 0.05), all tested drivers, with controls"

In [25]:
# CHECK — Section 2: your headline number, with its unit. Run it; you need not read it.
def check_headline(printed, in_units, unit, paper_number, paper_where, units_ok):
    """Is the unit right for this outcome, and is the in-unit number the printed one converted?"""
    if any(v is None for v in (printed, in_units, unit, paper_number, paper_where)):
        print("Not filled yet: set all five names in the cell above.")
        return False
    u = " ".join(str(unit).lower().split())
    if u not in units_ok:
        print(f"'{unit}' is not a unit this outcome is measured in. Allowed here: " + "; ".join(sorted(units_ok)) + ".")
        print("Ask what the outcome variable is: a log, a proportion (0 to 1), or a rate per 100,000.")
        return False
    wants = [f(float(printed)) for f in units_ok[u]]     # every accepted reading of this unit
    if not any(abs(float(in_units) - w) <= max(0.005, 0.02 * abs(w)) for w in wants):
        print(f"Your in-unit number does not follow from your printed estimate in '{unit}'.")
        print("Convert the printed coefficient again: what does one unit of the outcome mean here?")
        return False
    if not str(paper_where).strip() or len(str(paper_where).split()) < 2:
        print("Name where the paper's number is: table, column (or figure), and row.")
        return False
    print("Unit check passed. Now write the comparison in your report: your number, theirs, the gap, and its likely source.")
    return True

if RUNNING is None or H is None:
    BASE = None
else:
    below = dwi[(dwi[RUNNING] < CUTOFF) & (dwi[RUNNING] >= CUTOFF - H)]
    BASE = below["recidivism"].mean()                  # the mean just below the cutoff, within your bandwidth
HANSEN_UNITS = {
    "percentage points": [lambda b: 100 * b],
    "percent relative to the mean below the cutoff": [lambda b: 100 * b / BASE if BASE else float("nan")],
}
ok = check_headline(PRINTED, IN_UNITS, UNIT, PAPER_NUMBER, PAPER_WHERE, HANSEN_UNITS)

Unit check passed. Now write the comparison in your report: your number, theirs, the gap, and its likely source.


### Theirs vs mine

One row per number you replicate. The row label says where the number is in the paper (for example, `Table 2, col 3: effect on log wage`). `gap_pct` is computed for you as 100 × (mine − theirs) / |theirs|. This cell writes `output/headline_table.csv`, which `python reproduce.py` prints and your README quotes.

In [29]:
headline = pd.DataFrame(
    {
        "paper_estimate": pd.Series(dtype="float"),
        "my_estimate": pd.Series(dtype="float"),
        "gap_pct": pd.Series(dtype="float"),
        "likely_source": pd.Series(dtype="object"),
    },
    index=pd.Index([], name="estimate", dtype="object"),
)

# One line per row; take my_estimate from the cell that computed it, never type it in.
# headline.loc["Table 2, col 3: <what it measures>"] = [<paper's number>, <your variable>, np.nan, "<likely source>"]
headline.loc["Table 3, Panel A (bandwidth 0.05): effect of BAC over 0.08 on recidivism"] = [-0.021, JUMP, np.nan, "smaller sample; no county FE or prior-offense controls"]
headline["gap_pct"] = 100 * (headline["my_estimate"] - headline["paper_estimate"]) / headline["paper_estimate"].abs()
headline.to_csv(ROOT / "output" / "headline_table.csv")
headline

,paper_estimate,my_estimate,gap_pct,likely_source
estimate,,,,
"Table 3, Panel A (bandwidth 0.05): effect of BAC over 0.08 on recidivism",-0.021,-0.022163,-5.539657,smaller sample; no county FE or prior-offense ...


In [30]:
print(dwi.columns)

fit_nocontrols = smf.ols("recidivism ~ dui + x + dui:x", data=s).fit(cov_type="cluster", cov_kwds={"groups": s["low_score"]})
print(fit_nocontrols.params["dui"])

Index(['Date', 'Alcohol1', 'Alcohol2', 'low_score', 'male', 'white',
       'recidivism', 'acc', 'aged', 'year', 'bac1', 'bac2', 'bac_min'],
      dtype='object')
-0.021845285314709577


### Discrepancy log

One row per discrepancy, including the ones you could not explain. A clean match reported without investigation is less credible than a 3% gap that is explained.

| # | Estimate (row above) | What differs | Size of the gap | Likely source | How I checked it | Is the paper ambiguous here? |
|---|---|---|---|---|---|---|
| 1 | Table 3, Panel A | Mine -0.0222 vs paper -0.021 | 0.0012 | sample and controls| Same window, cutoff, kernel and clustering as the paper | No |
| 2 | Table 3, Panel A | My N 93899 vs paper 95111 | 1212 fewer stops | sample construction: public data likely cleaned differently | Same BAC window and years, but N still differs. | Yes, sample cleaning is not fully described |
| 3 | Table 3, Panel A | Paper also controls for county FE and prior offenses | -0.0218 with no controls vs -0.0222 | variable definition: not in the public data | Listed the columns; reran without controls, almost no change | No |
| 4 | Table 3, Panel A | Which BAC reading, bandwidth, kernel and controls | No gap | decision the paper makes | Paper uses the lower reading, so I use low_score, bandwidth 0.05, rectangular kernel, like same as Table 3 | No |

**Ambiguities in the paper, to take into Section 3:**

- How the sample was cleaned (why is my N different )
- The bandwidth is chosen by hand
- Clustering the standard errors by BAC bin